# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library and its Croissant schema.

### Dataset Source
**Croissant schema URL:** [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access and print the metadata (as an object)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\nPublished: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Authors: {[getattr(a, 'name', a['@id']) for a in getattr(metadata, 'author', [])]}")


## 2. Data Overview

Review available record sets, fields, and their IDs. All references are to entities by their `@id`, following ML Croissant conventions.

**Note:** Use the `.record_sets` property to enumerate the available record sets. Each record set exposes `@id` and its associated fields (columns), also via `@id`.

In [ ]:
# List all record sets and their fields (referenced by @id)
print('Available record sets:')
record_sets = []
for record_set in dataset.record_sets:
    print(f"- @id: {record_set.id} | name: {record_set.name}")
    record_sets.append(record_set.id)
    # Print fields for each record set
    print('  Fields:')
    for field in record_set.fields:
        print(f"    - @id: {field.id} | name: {field.name}  (dataType: {field.data_type})")
    print()

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames for further analysis. All record set and field accesses are by `@id` as listed above.

In [ ]:
# Collect all data into DataFrames referencing record set @ids and field @ids
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

        print(f'Columns for record set {record_set_id}:')
        print(df.columns.tolist())
        print(df.head(3), '\n')

# For this dataset, there is likely only one primary tabular record set. Let's pick the first one for EDA.
primary_record_set_id = record_sets[0] if record_sets else None

if primary_record_set_id is not None:
    print(f"\nPrimary record set for further EDA: {primary_record_set_id}")
else:
    print("No record sets found in this Croissant package.")

## 4. Exploratory Data Analysis (EDA)

Select a numeric field (by its `@id`) for analysis; demonstrate filtering, normalization, and grouping using only IDs in code.

In [ ]:
# Example: Identify a numeric field by its @id
if primary_record_set_id is not None:
    primary_df = dataframes[primary_record_set_id]

    # Let's auto-detect a suitable numeric field by checking float/integer fields from the schema
    numeric_field_id = None
    group_field_id = None
    for record_set in dataset.record_sets:
        if record_set.id == primary_record_set_id:
            for field in record_set.fields:
                if field.data_type and ("Float" in field.data_type or "Integer" in field.data_type or "Number" in field.data_type):
                    numeric_field_id = field.id
                    break
            # Pick a string/categorical field for grouping
            for field in record_set.fields:
                if field.data_type and ("Text" in field.data_type or "String" in field.data_type):
                    group_field_id = field.id
                    break
            break
    
    print(f"Using numeric_field_id: {numeric_field_id}")
    print(f"Using group_field_id: {group_field_id}")

    if numeric_field_id and numeric_field_id in primary_df.columns:
        # Ensure numeric type
        primary_df[numeric_field_id] = pd.to_numeric(primary_df[numeric_field_id], errors='coerce')

        threshold = primary_df[numeric_field_id].mean()  # use mean as an automatic threshold
        filtered_df = primary_df[primary_df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the field in the filtered subset
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a categorical field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field present in the primary record set.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

For this demonstration, we plot the distribution of the selected numeric field, and illustrate grouping if a categorical field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution if possible
if primary_record_set_id is not None and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(primary_df[numeric_field_id].dropna(), kde=True, bins=10, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group/categorical field is available, show group means
    if group_field_id and group_field_id in primary_df.columns:
        plt.figure(figsize=(10,6))
        group_means = primary_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)[:10]
        sns.barplot(x=group_means.values, y=group_means.index, orient='h', color='salmon')
        plt.title(f'Mean {numeric_field_id} by {group_field_id} (Top 10)')
        plt.xlabel(f'Mean {numeric_field_id}')
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion

- This notebook demonstrated the use of the `mlcroissant` library to load and explore a fully FAIR dataset described by a Croissant JSON-LD schema, referencing all schema elements by unique `@id`.
- We loaded all record sets and explored their structure, then extracted the main tabular data by `@id` and performed a simple exploratory and visualization workflow.
- This approach ensures precise, reproducible FAIR data access. You may now proceed to more advanced modeling or integrate this data into your own workflows.